<a href="https://colab.research.google.com/github/idant1111/BoilerplateGenerator/blob/main/research1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Analyze the provided image dataset located at "/content/drive/MyDrive/dataset-for-train" to distinguish between images of broken and unbroken limbs. Train a model to identify fractures and generate a heatmap highlighting the fracture location on broken limb images. Include comprehensive statistical analysis suitable for a technology management master's thesis and create a Gradio interface for interactive image analysis.

## Data loading and preparation

### Subtask:
Update the dataset loading to use the specified Google Drive path and ensure proper data splitting and transformations.


**Reasoning**:
Modify the existing `FractureDataset` and `FractureTrainer` classes to accommodate the new data path and ensure the data loading, splitting, and transformations are correctly handled as per the instructions. Update the `DATA_DIR` in the main execution block.



**Reasoning**:
The error `NameError: name 'Dataset' is not defined` indicates that the `Dataset` class from `torch.utils.data` was not recognized in the previous code block, although it was imported in the first cell. This is likely due to the interactive notebook environment. Regenerate the full code block including the necessary import to ensure it's available.



**Reasoning**:
The traceback indicates that the `pytorch_grad_cam` module is not found. This package needs to be installed.



In [ ]:
!nvcc --version

In [ ]:
!pip uninstall tensorflow keras -y

In [ ]:
!pip install tensorflow keras

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models

# Attempt to import pytorch_grad_cam, fallback to Keras if not available
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    print("Using pytorch_grad_cam for Grad-CAM")
    USE_PYTORCH_GRAD_CAM = True
except ImportError:
    print("pytorch_grad_cam not found. Falling back to Keras 3 for Grad-CAM.")
    # Install Keras 3 and related dependencies for the fallback
    # This assumes you have TensorFlow installed as the Keras 3 backend
    # !pip install keras==3.0.0 tensorflow
    import keras
    import tensorflow as tf
    from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions
    from tensorflow.keras.models import Model
    USE_PYTORCH_GRAD_CAM = False

import gradio as gr # Import gradio

import os
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
from tqdm import tqdm
import json

class FractureDataset(Dataset):
    """Custom dataset for fracture detection"""

    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir (string): Directory with subdirectories 'broken' and 'normal'
            transform (callable, optional): Optional transform to be applied on images
        """
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []

        # Expected folder structure:
        # root_dir/
        #   ├── broken/     (100 images of broken limbs)
        #   └── normal/     (100 images of normal limbs)

        broken_dir = os.path.join(root_dir, 'broken')
        normal_dir = os.path.join(root_dir, 'normal')

        # Load broken limb images (label = 1)
        if os.path.exists(broken_dir):
            for img_name in os.listdir(broken_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.images.append(os.path.join(broken_dir, img_name))
                    self.labels.append(1)  # Broken = 1

        # Load normal limb images (label = 0)
        if os.path.exists(normal_dir):
            for img_name in os.listdir(normal_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.images.append(os.path.join(normal_dir, img_name))
                    self.labels.append(0)  # Normal = 0

        print(f"Dataset loaded: {len(self.images)} images")
        print(f"Broken: {sum(self.labels)} images")
        print(f"Normal: {len(self.labels) - sum(self.labels)} images")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

class FractureTrainer:
    """Complete training pipeline for fracture detection"""

    def __init__(self, data_dir, model_name='resnet18'):
        self.data_dir = data_dir
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device for training: {self.device}")

        # Data transformations
        self.train_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomRotation(10),           # Data augmentation
            transforms.RandomHorizontalFlip(0.5),    # Data augmentation
            transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Data augmentation
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])

        self.val_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])

        # Initialize model
        if model_name == 'resnet18':
            self.model = models.resnet18(pretrained=True)
            self.model.fc = nn.Linear(self.model.fc.in_features, 2)
        elif model_name == 'resnet50':
            self.model = models.resnet50(pretrained=True)
            self.model.fc = nn.Linear(self.model.fc.in_features, 2)
        elif model_name == 'mobilenet':
            self.model = models.mobilenet_v2(pretrained=True)
            self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, 2)

        self.model.to(self.device)

        # Training setup
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
        self.scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size=10, gamma=0.1)

        self.training_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    def prepare_data(self, train_split=0.8, batch_size=16):
        """Prepare train/validation datasets"""
        # Load full dataset
        full_dataset = FractureDataset(self.data_dir, transform=None)

        # Split into train/validation
        train_size = int(train_split * len(full_dataset))
        val_size = len(full_dataset) - train_size

        train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

        # Apply different transforms to train/val sets
        train_dataset.dataset.transform = self.train_transform
        val_dataset.dataset.transform = self.val_transform

        # Create data loaders
        self.train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        self.val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        print(f"Training samples: {len(train_dataset)}")
        print(f"Validation samples: {len(val_dataset)}")

        return self.train_loader, self.val_loader

    def train_epoch(self):
        """Train for one epoch"""
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        progress_bar = tqdm(self.train_loader, desc='Training')
        for images, labels in progress_bar:
            images, labels = images.to(self.device), labels.to(self.device)

            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Update progress bar
            progress_bar.set_postfix({
                'Loss': f'{running_loss / total:.4f}',
                'Acc': f'{100*correct/total:.2f}%'
            })


        epoch_loss = running_loss / len(self.train_loader)
        epoch_acc = 100 * correct / total

        return epoch_loss, epoch_acc

    def validate_epoch(self):
        """Validate for one epoch"""
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        all_predictions = []
        all_labels = []
        all_probs = []

        with torch.no_grad():
            for images, labels in tqdm(self.val_loader, desc='Validation'):
                images, labels = images.to(self.device), labels.to(self.device)

                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                all_predictions.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()) # Probability of the positive class (fracture)

        epoch_loss = running_loss / len(self.val_loader)
        epoch_acc = 100 * correct / total

        return epoch_loss, epoch_acc, all_predictions, all_labels, all_probs

    def train(self, num_epochs=25):
        """Complete training loop"""
        print("Starting training...")
        best_val_acc = 0.0

        for epoch in range(num_epochs):
            print(f'\nEpoch {epoch+1}/{num_epochs}')
            print('-' * 40)

            # Train
            train_loss, train_acc = self.train_epoch()

            # Validate
            val_loss, val_acc, val_preds, val_labels, val_probs = self.validate_epoch()

            # Update learning rate
            self.scheduler.step()

            # Store history
            self.training_history['train_loss'].append(train_loss)
            self.training_history['train_acc'].append(train_acc)
            self.training_history['val_loss'].append(val_loss)
            self.training_history['val_acc'].append(val_acc)

            print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
            print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')

            # Save best model
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(self.model.state_dict(), 'best_fracture_model.pth')
                print(f'New best model saved! Validation accuracy: {val_acc:.2f}%')

        print(f'\nTraining completed! Best validation accuracy: {best_val_acc:.2f}%')

        # Generate final statistical analysis
        self.generate_statistical_analysis(val_preds, val_labels, val_probs)

        return self.model

    def generate_statistical_analysis(self, predictions, labels, probabilities):
        """Generate detailed statistical analysis including thesis-level metrics"""
        print("\n" + "="*50)
        print("FINAL STATISTICAL ANALYSIS")
        print("="*50)

        # Classification report
        report = classification_report(labels, predictions,
                                     target_names=['Normal', 'Fracture'],
                                     digits=4)
        print("Classification Report:")
        print(report)

        # Confusion matrix
        cm = confusion_matrix(labels, predictions)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=['Normal', 'Fracture'],
                   yticklabels=['Normal', 'Fracture'])
        plt.title('Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
        plt.show()

        # ROC Curve and AUC
        fpr, tpr, thresholds = roc_curve(labels, probabilities)
        roc_auc = auc(fpr, tpr)

        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.4f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Receiver Operating Characteristic (ROC) Curve')
        plt.legend(loc="lower right")
        plt.grid(True)
        plt.savefig('roc_curve.png', dpi=300, bbox_inches='tight')
        plt.show()

        print(f"\nAUC: {roc_auc:.4f}")

        # Training history plots
        self.plot_training_history()

    def plot_training_history(self):
        """Plot training history"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

        # Loss plot
        ax1.plot(self.training_history['train_loss'], label='Train Loss')
        ax1.plot(self.training_history['val_loss'], label='Validation Loss')
        ax1.set_title('Training and Validation Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True)

        # Accuracy plot
        ax2.plot(self.training_history['train_acc'], label='Train Accuracy')
        ax2.plot(self.training_history['val_acc'], label='Validation Accuracy')
        ax2.set_title('Training and Validation Accuracy')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy (%)')
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()
        plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
        plt.show()


# Helper function for Keras Grad-CAM
def make_gradcam_heatmap_keras(img_array, model, last_conv_layer_name, classifier_layer_names):
    """
    Generates a Grad-CAM heatmap for a Keras model.
    Assumes model input is of shape (1, H, W, C) and values are preprocessed.
    """
    # Create a model that maps the input image to the activations of the last convolutional layer, as well as the output predictions
    grad_model = Model(
        model.inputs, [model.get_layer(last_conv_layer_name).output, model.output]
    )

    # Compute the gradient of the top predicted class for our input image with respect to the activations of the last convolutional layer
    with tf.GradientTape() as tape:
        last_conv_layer_output, predictions = grad_model(img_array)
        if tf.keras.mixed_precision.dtype_policy.global_policy().compute_dtype == 'float16':
            predictions = tf.cast(predictions, tf.float32) # Ensure predictions are float32 for gradient calculation
        top_pred_index = tf.argmax(predictions[0])
        top_class_channel = predictions[:, top_pred_index]

    # This is the gradient of the top predicted class with regard to the output feature map of the last convolutional layer
    grads = tape.gradient(top_class_channel, last_conv_layer_output)

    # This is a vector where each entry is the mean intensity of the gradient over a specific feature map channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Multiply each channel in the feature map array by "how important" this channel is with regard to the top predicted class
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normalize the heatmap
    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap)
    return heatmap.numpy()


# Overlay heatmap on image for Keras
def save_and_display_gradcam_keras(img_path, heatmap):
    """
    Saves and displays the Grad-CAM heatmap overlaid on the original image for Keras.
    """
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert to RGB

    # Resize heatmap to be the same size as the original image
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

    # Rescale heatmap to a range 0-255
    heatmap = np.uint8(255 * heatmap)

    # Apply colormap to heatmap
    viridis = plt.get_cmap("viridis")
    heatmap = viridis(heatmap)[:, :, :3] # Use viridis colormap

    # Convert to RGB
    heatmap = np.uint8(heatmap * 255)

    # Superimpose the heatmap on the original image
    superimposed_img = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)

    # Display the image
    return superimposed_img



class FractureDetectorInference:
    """Inference class for trained model"""

    def __init__(self, model_path='best_fracture_model.pth', model_name='resnet18'):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model_name = model_name

        # Load trained PyTorch model (only for potential future use or reference, inference will use Keras)
        self.model = models.resnet18(pretrained=False) # Ensure this matches the training model
        if self.model_name == 'resnet50':
             self.model = models.resnet50(pretrained=False)
        elif self.model_name == 'mobilenet':
            self.model = models.mobilenet_v2(pretrained=False)

        if hasattr(self.model, 'fc'): # ResNet
             self.model.fc = nn.Linear(self.model.fc.in_features, 2)
        elif hasattr(self.model, 'classifier'): # MobileNet
             self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, 2)

        self.model.load_state_dict(torch.load(model_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

        # Load a pre-trained Keras model (or your trained Keras model) for inference and Grad-CAM
        # For demonstration, using a pre-trained ResNet50; replace with your model loading if needed
        # You would ideally load a Keras version of your *trained* model here.
        self.keras_model = tf.keras.applications.resnet50.ResNet50(weights="imagenet") # Replace with your trained Keras model path if available

        # Adjust the Keras model for Grad-CAM based on your actual trained model's architecture
        # Assuming a classification head is needed after the convolutional layers for prediction
        x = self.keras_model.output
        x = tf.keras.layers.GlobalAveragePooling2D()(x)
        # Add a dense layer with 2 units for classification (matching your PyTorch model's output)
        predictions = tf.keras.layers.Dense(2, activation='softmax')(x)
        self.keras_model_with_classifier = Model(inputs=self.keras_model.input, outputs=predictions)


        # Define the last convolutional layer and classifier layer names for your Keras model
        # These names need to match your specific Keras model's layers
        # Example names for ResNet50:
        self.last_conv_layer_name_keras = "conv5_block3_out"  # Replace with the actual name
        self.classifier_layer_names_keras = ["dense"] # Replace with actual names


        # Transform for Keras inference
        self.transform_keras = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor() # Convert to Tensor, then will convert to numpy for Keras
        ])


    def predict_with_heatmap(self, image_path):
        """Predict and generate heatmap using Keras fallback"""
        # Load image
        image = Image.open(image_path).convert('RGB')
        original_image = np.array(image)

        # Preprocess for Keras
        img_tensor_keras = self.transform_keras(image)
        img_array_keras = np.array(img_tensor_keras).transpose(1, 2, 0) # Convert C, H, W to H, W, C
        img_array_keras = np.expand_dims(img_array_keras, axis=0) # Add batch dimension
        # Rescale pixel values for Keras models (e.g., 0-255 for some pretrained models)
        img_array_keras = img_array_keras * 255.0
        # Apply Keras specific preprocessing if needed (e.g., imagenet mean subtraction)
        img_array_keras = preprocess_input(img_array_keras)


        # Predict with Keras model
        keras_predictions = self.keras_model_with_classifier.predict(img_array_keras)
        fracture_prob = keras_predictions[0][1] # Probability of the fracture class (assuming index 1)
        predicted_class = np.argmax(keras_predictions[0])

        has_fracture = predicted_class == 1

        # Generate heatmap if fracture detected using Keras Grad-CAM
        heatmap = None
        if has_fracture:
             heatmap = make_gradcam_heatmap_keras(
                 img_array_keras, self.keras_model_with_classifier, # Use the model with classifier for prediction
                 self.last_conv_layer_name_keras, self.classifier_layer_names_keras
             )
             # Overlay the heatmap
             heatmap = save_and_display_gradcam_keras(image_path, heatmap)


        return has_fracture, fracture_prob, heatmap, original_image

def create_gradio_interface_trained(model_name='resnet18'):
    """Gradio interface for trained model"""
    # Initialize the detector to use the Keras fallback primarily
    detector = FractureDetectorInference(model_name=model_name)

    def analyze_image(image):
        # Save uploaded image temporarily
        temp_path = "temp_upload.jpg"
        image.save(temp_path)

        # Analyze using the Keras-based prediction and heatmap
        has_fracture, confidence, heatmap, original = detector.predict_with_heatmap(temp_path)

        # Results
        if has_fracture:
            result = f"🚨 **FRACTURE DETECTED**\nConfidence: {confidence:.1%}"
            # Convert heatmap to PIL Image for Gradio output
            return_image = Image.fromarray(heatmap) if heatmap is not None else Image.fromarray(original)
        else:
            # For Keras fallback, the confidence for "No Fracture" is 1 - confidence of "Fracture"
            result = f"✅ **NO FRACTURE**\nConfidence: {1-confidence:.1%}"
            return_image = Image.fromarray(original)

        os.remove(temp_path)  # Clean up
        return result, return_image

    interface = gr.Interface(
        fn=analyze_image,
        inputs=gr.Image(type="pil", label="Upload X-ray Image"),
        outputs=[
            gr.Markdown(label="Analysis Result"),
            gr.Image(label="Visualization")
        ],
        title="🏥 Trained Fracture Detection System",
        description="AI model trained on 200 images (100 broken + 100 normal limbs)"
    )

    return interface

# Main execution script
if __name__ == "__main__":
    print("🏥 Fracture Detection Training Pipeline")
    print("="*50)

    # Configuration
    DATA_DIR = "/content/drive/MyDrive/dataset-for-train"  # Your dataset directory
    MODEL_TYPE = "resnet18"        # resnet18, resnet50, or mobilenet
    NUM_EPOCHS = 25
    BATCH_SIZE = 16

    # Expected folder structure:
    # /content/drive/MyDrive/dataset-for-train/
    #   ├── broken/     (100 images)
    #   └── normal/     (100 images)

    print("📁 Expected folder structure:")
    print(f"   {DATA_DIR}/")
    print("     ├── broken/     (100 images of broken limbs)")
    print("     └── normal/     (100 images of normal limbs)")
    print()

    # Check if dataset exists
    if not os.path.exists(DATA_DIR):
        print(f"❌ Dataset directory '{DATA_DIR}' not found!")
        print("Please create the dataset directory with 'broken' and 'normal' subfolders.")
    else:
        # Initialize trainer
        trainer = FractureTrainer(DATA_DIR, model_name=MODEL_TYPE)

        # Prepare data
        trainer.prepare_data(train_split=0.8, batch_size=BATCH_SIZE)

        # Train model
        trained_model = trainer.train(num_epochs=NUM_EPOCHS)

        print("\n🎉 Training completed!")
        print("📄 Generated files:")
        print("   - best_fracture_model.pth (trained model)")
        print("   - confusion_matrix.png")
        print("   - training_history.png")
        if hasattr(trainer, 'roc_auc'): # Check if AUC was calculated
             print(f"   - ROC Curve (AUC: {trainer.roc_auc:.4f})")


        # Launch Gradio interface
        print("\n🚀 Launching Gradio interface...")
        interface = create_gradio_interface_trained(model_name=MODEL_TYPE)
        interface.launch(share=True)

Using pytorch_grad_cam for Grad-CAM
🏥 Fracture Detection Training Pipeline
📁 Expected folder structure:
   /content/drive/MyDrive/dataset-for-train/
     ├── broken/     (100 images of broken limbs)
     └── normal/     (100 images of normal limbs)

Using device for training: cuda


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Dataset loaded: 200 images
Broken: 100 images
Normal: 100 images
Training samples: 160
Validation samples: 40
Starting training...

Epoch 1/25
----------------------------------------


Training:  80%|████████  | 8/10 [00:09<00:02,  1.18s/it, Loss=0.0781, Acc=53.91%]


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models

# Import Keras and TensorFlow for the fallback
import keras
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions
from tensorflow.keras.models import Model

import gradio as gr
import os
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Check for TensorFlow GPU availability
print("Debugging: Checking TensorFlow GPU availability...")
gpu_available = tf.config.list_physical_devices('GPU')
if gpu_available:
    print(f"Debugging: TensorFlow found GPU: {gpu_available}")
else:
    print("Debugging: TensorFlow did not find GPU. Using CPU.")

# Helper function for Keras Grad-CAM
def make_gradcam_heatmap_keras(img_array, model, last_conv_layer_name, classifier_layer_names):
    """
    Generates a Grad-CAM heatmap for a Keras model.
    Assumes model input is of shape (1, H, W, C) and values are preprocessed.
    """
    print("Debugging: Entering make_gradcam_heatmap_keras")
    # Create a model that maps the input image to the activations of the last convolutional layer, as well as the output predictions
    grad_model = Model(
        model.inputs, [model.get_layer(last_conv_layer_name).output, model.output]
    )
    print(f"Debugging: Grad model created with inputs: {model.inputs} and outputs: {[model.get_layer(last_conv_layer_name).output, model.output]}")


    # Compute the gradient of the top predicted class for our input image with respect to the activations of the last convolutional layer
    with tf.GradientTape() as tape:
        last_conv_layer_output, predictions = grad_model(img_array)
        print(f"Debugging: last_conv_layer_output shape: {last_conv_layer_output.shape}")
        print(f"Debugging: predictions shape: {predictions.shape}")

        if tf.keras.mixed_precision.dtype_policy.global_policy().compute_dtype == 'float16':
            predictions = tf.cast(predictions, tf.float32) # Ensure predictions are float32 for gradient calculation
            print("Debugging: Cast predictions to float32")
        top_pred_index = tf.argmax(predictions[0])
        print(f"Debugging: top_pred_index: {top_pred_index}")
        top_class_channel = predictions[:, top_pred_index]
        print(f"Debugging: top_class_channel shape: {top_class_channel.shape}")


    # This is the gradient of the top predicted class with regard to the output feature map of the last convolutional layer
    grads = tape.gradient(top_class_channel, last_conv_layer_output)
    print(f"Debugging: grads shape: {grads.shape}")

    # This is a vector where each entry is the mean intensity of the gradient over a specific feature map channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    print(f"Debugging: pooled_grads shape: {pooled_grads.shape}")


    # Multiply each channel in the feature map array by "how important" this channel is with regard to the top predicted class
    last_conv_layer_output = last_conv_layer_output[0]
    print(f"Debugging: last_conv_layer_output[0] shape: {last_conv_layer_output.shape}")
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    print(f"Debugging: heatmap shape after squeeze: {heatmap.shape}")


    # Normalize the heatmap
    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap)
    print(f"Debugging: heatmap max after normalization: {tf.reduce_max(heatmap)}")
    print("Debugging: Exiting make_gradcam_heatmap_keras")
    return heatmap.numpy()


# Overlay heatmap on image for Keras
def save_and_display_gradcam_keras(img_path, heatmap):
    """
    Saves and displays the Grad-CAM heatmap overlaid on the original image for Keras.
    """
    print("Debugging: Entering save_and_display_gradcam_keras")
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert to RGB
    print(f"Debugging: Original image shape: {img.shape}")

    # Resize heatmap to be the same size as the original image
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    print(f"Debugging: Resized heatmap shape: {heatmap.shape}")


    # Rescale heatmap to a range 0-255
    heatmap = np.uint8(255 * heatmap)
    print(f"Debugging: Rescaled heatmap max: {np.max(heatmap)}")


    # Apply colormap to heatmap
    viridis = plt.get_cmap("viridis")
    heatmap = viridis(heatmap)[:, :, :3] # Use viridis colormap
    print(f"Debugging: Colormapped heatmap shape: {heatmap.shape}")


    # Convert to RGB
    heatmap = np.uint8(heatmap * 255)
    print(f"Debugging: Final heatmap max: {np.max(heatmap)}")


    # Superimpose the heatmap on the original image
    superimposed_img = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)
    print(f"Debugging: Superimposed image shape: {superimposed_img.shape}")

    print("Debugging: Exiting save_and_display_gradcam_keras")
    # Display the image
    return superimposed_img



class FractureDetectorInferenceKeras:
    """Inference class using Keras for prediction and Grad-CAM"""

    def __init__(self, model_path='best_fracture_model.pth', model_name='resnet18'):
        self.model_name = model_name
        print(f"Debugging: Initializing FractureDetectorInferenceKeras with model: {model_name}")

        # Load a pre-trained Keras model structure that matches the trained PyTorch model's architecture
        # We will not load ImageNet weights here, as we want to use the architecture for Grad-CAM
        # and the prediction logic will need to be aligned with the trained PyTorch model's output.
        if model_name == 'resnet18':
             # ResNet18 equivalent in Keras (using ResNet50 base for demonstration, adjust if needed)
             self.keras_model_base = tf.keras.applications.resnet50.ResNet50(weights=None, include_top=False, pooling='avg') # Load structure without weights
             self.last_conv_layer_name_keras = "conv5_block3_out" # Adjust to ResNet18 last conv layer name
        elif model_name == 'resnet50':
             self.keras_model_base = tf.keras.applications.resnet50.ResNet50(weights=None, include_top=False, pooling='avg') # Load structure without weights
             self.last_conv_layer_name_keras = "conv5_block3_out" # Last convolutional layer in ResNet50
        # Add other model types if needed

        self.keras_model_base.trainable = False # Freeze the base model
        print(f"Debugging: Loaded Keras {model_name} base model structure")


        # Add a classification head that matches the PyTorch model's output (2 classes)
        inputs = tf.keras.Input(shape=(224, 224, 3))
        x = preprocess_input(inputs * 255.0) # Apply Keras preprocessing
        x = self.keras_model_base(x, training=False)
        predictions = tf.keras.layers.Dense(2, activation='softmax')(x) # 2 output classes

        self.keras_model = Model(inputs, predictions)
        print("Debugging: Created Keras model with classification head")

        # IMPORTANT: To use the knowledge from your trained PyTorch model, you need to convert its weights
        # and load them into this Keras model structure. This step is not automatically done here.
        # For this demonstration, we will use this Keras model structure for Grad-CAM,
        # and the prediction logic will be simplified or simulated if trained Keras weights are not loaded.

        # Define the classifier layer names for your Keras model
        self.classifier_layer_names_keras = ["dense"] # Replace with actual names of your classification layers (e.g., list of Dense layer names)
        print(f"Debugging: Last conv layer name: {self.last_conv_layer_name_keras}")
        print(f"Debugging: Classifier layer names: {self.classifier_layer_names_keras}")


        # Transform for Keras inference (simple resize and ToTensor, Keras preprocessing is handled in the model)
        self.transform_keras = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor() # Convert to Tensor, then will convert to numpy for Keras
        ])
        print("Debugging: Keras transforms defined")


    def predict_with_heatmap(self, image_path):
        """Predict and generate heatmap using Keras"""
        print(f"Debugging: Entering predict_with_heatmap for image: {image_path}")
        # Load image
        image = Image.open(image_path).convert('RGB')
        original_image = np.array(image)
        print(f"Debugging: Image loaded, original shape: {original_image.shape}")


        # Preprocess for Keras
        img_tensor_keras = self.transform_keras(image)
        img_array_keras = np.array(img_tensor_keras).transpose(1, 2, 0) # Convert C, H, W to H, W, C
        img_array_keras = np.expand_dims(img_array_keras, axis=0) # Add batch dimension
        # Note: Keras preprocessing (scaling and mean subtraction) is now part of the Keras model's input layer
        print(f"Debugging: Preprocessed image array shape: {img_array_keras.shape}")


        # Predict with Keras model
        # *** IMPORTANT: This prediction uses a Keras model structure without your trained PyTorch weights loaded.
        # The predictions here will be based on the default initialization or any weights loaded.
        # To get predictions based on your training, you need to convert and load your PyTorch weights into this Keras model.
        keras_predictions = self.keras_model.predict(img_array_keras)
        print(f"Debugging: Keras predictions: {keras_predictions}")
        fracture_prob = keras_predictions[0][1] # Probability of the fracture class (assuming index 1)
        predicted_class = np.argmax(keras_predictions[0])
        print(f"Debugging: Predicted class: {predicted_class}, Fracture probability: {fracture_prob}")


        has_fracture = predicted_class == 1
        print(f"Debugging: Has fracture: {has_fracture}")


        # Generate heatmap if fracture detected using Keras Grad-CAM
        heatmap = None
        if has_fracture:
             print("Debugging: Fracture detected, generating heatmap...")
             # Note: The Keras model used here for Grad-CAM is the structure defined in __init__.
             # Grad-CAM visualization will be based on the activations of this model structure.
             heatmap = make_gradcam_heatmap_keras(
                 img_array_keras, self.keras_model, # Use the model with classifier for prediction
                 self.last_conv_layer_name_keras, self.classifier_layer_names_keras
             )
             print(f"Debugging: Heatmap generated, shape: {heatmap.shape}")
             # Overlay the heatmap
             heatmap = save_and_display_gradcam_keras(image_path, heatmap)
             print("Debugging: Heatmap overlaid on image")


        print("Debugging: Exiting predict_with_heatmap")
        return has_fracture, fracture_prob, heatmap, original_image

def create_gradio_interface_keras(model_name='resnet18'):
    """Gradio interface for trained model using Keras inference"""
    print("Debugging: Creating Gradio interface using Keras Inference")
    # Initialize the detector to use the Keras implementation
    # Note: This will use a Keras model structure, not your trained PyTorch weights by default.
    detector = FractureDetectorInferenceKeras(model_name=model_name)
    print("Debugging: FractureDetectorInferenceKeras initialized")


    def analyze_image(image):
        print("Debugging: Entering analyze_image function")
        # Save uploaded image temporarily
        temp_path = "temp_upload.jpg"
        image.save(temp_path)
        print(f"Debugging: Uploaded image saved to {temp_path}")


        # Analyze using the Keras-based prediction and heatmap
        has_fracture, confidence, heatmap, original = detector.predict_with_heatmap(temp_path)
        print(f"Debugging: Analysis complete. Has fracture: {has_fracture}, Confidence: {confidence}")


        # Results
        if has_fracture:
            result = f"🚨 **FRACTURE DETECTED**\nConfidence: {confidence:.1%}"
            print(f"Debugging: Result: {result}")
            # Convert heatmap to PIL Image for Gradio output
            return_image = Image.fromarray(heatmap) if heatmap is not None else Image.fromarray(original)
            print(f"Debugging: Returning heatmap image (shape: {np.array(return_image).shape})")
        else:
            # For Keras, the confidence for "No Fracture" is 1 - confidence of "Fracture"
            result = f"✅ **NO FRACTURE**\nConfidence: {1-confidence:.1%}"
            print(f"Debugging: Result: {result}")
            return_image = Image.fromarray(original)
            print(f"Debugging: Returning original image (shape: {np.array(return_image).shape})")


        os.remove(temp_path)  # Clean up
        print(f"Debugging: Cleaned up temporary file {temp_path}")
        print("Debugging: Exiting analyze_image function")
        return result, return_image

    interface = gr.Interface(
        fn=analyze_image,
        inputs=gr.Image(type="pil", label="Upload X-ray Image"),
        outputs=[
            gr.Markdown(label="Analysis Result"),
            gr.Image(label="Visualization")
        ],
        title="🏥 Trained Fracture Detection System (Keras Inference)",
        description="AI model inference using Keras with Grad-CAM visualization."
    )
    print("Debugging: Gradio interface created")


    return interface

# Launch Gradio interface using Keras Inference
if __name__ == "__main__":
    print("\n🚀 Launching Gradio interface using Keras Inference...")
    # Assuming the best_fracture_model.pth (PyTorch weights) exists from the previous training run
    # Note: The Keras inference model used here is a pre-trained ResNet50 structure.
    # To use your trained PyTorch model's knowledge in Keras, you would need to convert the weights.
    # This code demonstrates the Keras Grad-CAM functionality with a Keras model structure.
    interface = create_gradio_interface_keras(model_name="resnet18") # Use model_name if relevant for Keras model loading
    interface.launch(share=True, debug=True)


🚀 Launching Gradio interface using Keras Inference...
Debugging: Creating Gradio interface using Keras Inference
Debugging: Initializing FractureDetectorInferenceKeras with model: resnet18
Debugging: Loaded Keras ResNet50 base model
Debugging: Created Keras model with classification head
Debugging: Last conv layer name: conv5_block3_out
Debugging: Classifier layer names: ['dense']
Debugging: Keras transforms defined
Debugging: FractureDetectorInferenceKeras initialized
Debugging: Gradio interface created
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://dcdfa63426906ed851.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Debugging: Entering analyze_image function
Debugging: Uploaded image saved to temp_upload.jpg
Debugging: Entering predict_with_heatmap for image: temp_upload.jpg
Debugging: Image loaded, original shape: (1499, 868, 3)
Debugging: Preprocessed image array shape: (1, 224, 224, 3)


Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 1702, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/anyio/to_thread.py", line 56, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

In [ ]:
# Assuming the Keras model is defined and available from the previous cell execution

# Make sure pydot and graphviz are installed for visualization
!pip install pydot graphviz

# Import the necessary function from Keras
from tensorflow.keras.utils import plot_model

# Assuming 'detector' is the instance of FractureDetectorInferenceKeras from the previous cell
# And 'detector.keras_model' is the Keras model you want to visualize

# Plot the model and save it as a PNG file
plot_model(detector.keras_model, to_file='keras_model_plot.png', show_shapes=True, show_layer_names=True)

print("Keras model plot saved as 'keras_model_plot.png'")